# 实验 2：压缩时保留失败、约束和待办

> 状态：verified；人工结构化事件，确定性抽取，无 LLM 摘要。

压缩要检查丢了什么。这里让旧的“测试通过”被后续真实失败推翻，并把关键待办放在长日志尾部。对照算法只读取前 500 个字节，改进算法逐条抽取最新结构化事实。正文：[优化策略](../02-patterns/02-optimization-strategies.md)。

In [1]:
import sys
from pathlib import Path
sys.path.insert(0,str(Path.cwd()))
from context_lab import compact,recall,serialized,byte_tokens
events=[
 {"id":"e1","kind":"constraint","key":"allow_restart","value":False},
 {"id":"e2","kind":"observation","key":"test_passed","value":True},
 *[{"id":f"log-{i}","kind":"log","text":"重复进度消息：仍在读取日志"} for i in range(40)],
 {"id":"e3","kind":"observation","key":"test_passed","value":False},
 {"id":"e4","kind":"pending","key":"next_step","value":"排查供电连接"},
]
expected={"allow_restart":False,"test_passed":False,"next_step":"排查供电连接"}
summary=compact(events)
summary

Out[1]: 
{'facts': {'allow_restart': False,
  'test_passed': False,
  'next_step': '排查供电连接'},
 'sources': {'allow_restart': 'e1', 'test_passed': 'e3', 'next_step': 'e4'},
 'method': 'structured-extract-v1',
 'lossy': True}


`test_passed` 必须为 False，否则压缩把一次失败变成了长期污染。这里“最新”由输入事件顺序决定；生产系统需有受信时间/版本，不能让无权来源用较新时间覆盖系统约束。

In [2]:
prefix=[]
for event in events:
    if byte_tokens(serialized(prefix+[event]))>500:
        break
    prefix.append(event)
naive=compact(prefix)
print({"raw_bytes":byte_tokens(serialized(events)),"naive_bytes":byte_tokens(serialized(naive)),"structured_bytes":byte_tokens(serialized(summary)),"naive_fact_recall":recall(naive,expected),"structured_fact_recall":recall(summary,expected)})
assert recall(summary,expected)==1
assert summary["facts"]["test_passed"] is False
assert summary["sources"]["test_passed"]=="e3"

{'raw_bytes': 3388, 'naive_bytes': 150, 'structured_bytes': 201, 'naive_fact_recall': 0.3333333333333333, 'structured_fact_recall': 1.0}


In [3]:
source_by_id={e["id"]:e for e in events}
for key,source_id in summary["sources"].items():
    assert source_by_id[source_id]["value"]==summary["facts"][key]
print("每个保留事实均可回读原始事件；丢弃的日志仍由原始事件列表保存。")

每个保留事实均可回读原始事件；丢弃的日志仍由原始事件列表保存。


**读结果**：事实召回率只覆盖这里预先标注的三个字段，未标注信息可能已经丢失。压缩比不能证明语言摘要保真。本例中的结构化抽取也不能直接作用于任意聊天记录：先要可靠识别事实、否定、来源和状态，再评测这一步的错误。源代码：[context_lab.py](context_lab.py)。